# nb08 — Figures 4 and 5: unharmonized | harmonized pairs of the recommended approaches

**Plan:** `article_1_revision_implementation_plan_261029.md` §18 (B1). **Decision:** Daniil,
2026-11-08 (`gates_review_261102.md` §9.4, B1 option b, "even now"): the panels of the submitted
Figures 4 and 5 (MNN, FSMVN, AMDBNorm next to SVA, with dendrograms) are replaced by the
recommended approaches of the four decision-tree leaves, each drawn next to the unharmonized data
of the same strategy and imputation. The figure count stays the same (rule "nothing grows").

**What is drawn.**

| Panel | Approach (strategy · imputation · method) | Why this one |
|---|---|---|
| 4A | S0 · No removal, no imputation, unharmonized | the starting point: batch dominates |
| 4B | J · FF only, softImpute, Angel | first recommended approach of the FF-only leaf |
| 4C | K · FFPE only, softImpute, SVA | the recommended approach of the FFPE-only leaf |
| 4D | C · RNA-seq only, KNN, SVA | first recommended approach of the RNA-seq-only leaf |
| 5A | D · Malignant samples only, KNN, FSQN R | first recommended approach of the multi-platform leaf |
| 5B | D · Malignant samples only, no imputation, ComBat | alternative (covariate method) |
| 5C | I · Rare batches removed, KNN, M-ComBat | alternative (reference-batch ComBat) |

Each panel B–D / A–C: four columns (PCA unharmonized, PCA harmonized, tSNE unharmonized, tSNE
harmonized), rows coloured by batch and by diagnosis (Figure 5: also by platform). The harmonized
column title carries the marker correlation and the cross-batch prediction of the approach (the
values of Table 3). Every approach is drawn without post-removal (all seven are post-removal = no).

**Input.** The embeddings cached by `nb07_selected_approaches_embeddings.ipynb` in `cache/nb07/`
(`{strat}__{imp}__{method}__post0.pkl`: matrix → genes with any missing value dropped →
standardised → PCA 50 → UMAP / tSNE on the PCs, seed 42). Nothing is recomputed here; a missing
pickle stops the notebook with the nb07 command to run.

**Outputs.** `figures_svg/fig4{A,B,C,D}_*.{svg,pdf,png}`, `figures_svg/fig5{A,B,C}_*.{svg,pdf,png}`,
`output/wp9/nb08_figure_4_5_panels.tsv` (panel → approach, samples, genes, scores).


## §0 Configuration

### §0a Imports

In [ ]:
# Imports. revision_lib is used only for plain-word names and paths (plan §1.1).
import hashlib
import textwrap

import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import revision_lib as rl

### §0b Paths and switches

| Switch | Default | Meaning |
|---|---|---|
| `PANELS` | 7 rows (table above) | panel → (strategy, imputation, method) |
| `PLATFORM_ROW_FIGURES` | `{"5"}` | figures with a third row coloured by platform |
| `POINT_SIZE`, `POINT_ALPHA` | 4, 0.6 | scatter points |
| `GLOBAL_FONT_SIZE` | 13 | legible at full-page width |

In [ ]:
# Paths and switches.
FIGURES_DIR = rl.REVISION_DIR / "figures_svg"
TABLES_DIR = rl.OUTPUT_DIR / "wp9"
CACHE_DIR = rl.REVISION_DIR / "cache/nb07"
SCORES_PATH = rl.OUTPUT_DIR / "wp3/four_scores.parquet"
SELECTION_PATH = rl.OUTPUT_DIR / "wp4/selection_after_visual_check.tsv"

# panel → (strategy, imputation, method); post-removal is off for all seven
PANELS = {
    "4A": ("S0_no_removal", "strict", "01_raw"),
    "4B": ("J_ff_only", "softimpute", "25_angel"),
    "4C": ("K_ffpe_only", "softimpute", "04_sva"),
    "4D": ("C_rnaseq_only", "knn", "04_sva"),
    "5A": ("D_malignant_only", "knn", "16_fsqn_r"),
    "5B": ("D_malignant_only", "strict", "05_combat"),
    "5C": ("I_rare_batches_removed", "knn", "29_combat_ref"),
}
PANEL_FILE_STEMS = {
    "4A": "fig4A_unharmonized_no_removal",
    "4B": "fig4B_ff_only_angel",
    "4C": "fig4C_ffpe_only_sva",
    "4D": "fig4D_rnaseq_only_sva",
    "5A": "fig5A_malignant_only_fsqn_r",
    "5B": "fig5B_malignant_only_combat",
    "5C": "fig5C_rare_batches_removed_mcombat",
}
PLATFORM_ROW_FIGURES = {"5"}
POINT_SIZE = 0.8  # pt² at print size (7,000 points in a ~40 × 25 mm scatter)
POINT_ALPHA = 0.6
SEED = 42
# Printed font sizes: every panel is drawn at its true size on the 180 × 275 mm page (D2), so
# 7 pt stays 7 pt; 6 pt is the smallest text (journal minimum).
GLOBAL_FONT_SIZE = 7
SMALL_FONT_SIZE = 6
MM = 1 / 25.4  # inches per mm
# Page slots (width, height) in mm, 2026-11-09 legibility fix (`nar_review_tools.py figures`:
# the 41-cm-wide panels printed at 3.4–4.6 pt). Figure 4 = 4A + 4B–4D + shared legend
# (40 + 3 × 62 + 42 = 268 mm); Figure 5 = 5A–5C + shared legend (3 × 72 + 54 = 270 mm).
# The legends are one file per page because 19–29 batch names do not fit next to each panel.
PANEL_SLOTS_MM = {
    "4A": (180, 40), "4B": (180, 62), "4C": (180, 62), "4D": (180, 62),
    "5A": (180, 72), "5B": (180, 72), "5C": (180, 72),
}
LEGEND_SLOTS_MM = {"4": (180, 42), "5": (180, 54)}
LEGEND_FILE_STEMS = {"4": "fig4_legend", "5": "fig5_legend"}

### §0c Palettes and plain-word names

Batch, diagnosis and strategy palettes copied from `nb07_selected_approaches_embeddings.ipynb`
§0c (the Article 1 palettes: Temperature-Split, tumours warm, normal B cells cold). A level
without a colour gets a deterministic MD5 colour, printed once.

In [ ]:
# Article 1 palettes (copied, not imported)
import matplotlib.colors as mcolors

lymphoma_ontogeny_palette = {
    "Bone_marrow_CD19+": "blue",
    "Immature": "#08519c",
    "Naive": "#4292c6",
    "GC": "#006d2c",
    "Centroblast": "#41ab5d",
    "Centrocyte": "#a1d99b",
    "B_cells": "lawngreen",
    "MZ": "olive",
    "Memory": "#00ced1",
    "Plasmablast": "#5f9ea0",
    "Plasma": "grey",
    "Follicular_Lymphoma": "#6a0dad",
    "Follicular_Lymphoma_GCB": "#9400d3",
    "Diffuse_Large_B_Cell_Lymphoma_ABC": "#ff4500",
    "Diffuse_Large_B_Cell_Lymphoma_GCB": "#d62728",
    "Diffuse_Large_B_Cell_Lymphoma": "#8b0000",
    "High_Grade_B_Cell_Lymphoma": "#ff00ff",
    "Burkitt_Lymphoma": "orange",
}

rna_batch_palette = {
    "RNASeq_FF_Total": "#1f77b4",
    "RNASeq_FF_PolyA": "#17becf",
    "RNASeq_FFPE_Exome_capture": "#9467bd",
    "RNASeq_FF_rRNADepletion": "#000080",
    "RNASeq_FFPE_PolyA": "#8c564b",
    "RNASeq_FF_Unknown": "#bcbd22",
    "RNASeq_FF_Exome_capture": "#2ca02c",
    "GPL96+97_FF_Unknown": "#d62728",
    "GPL570_FF_Unknown": "#ff7f0e",
    "GPL570_Unknown_Unknown": "#ffbb78",
    "GPL570_FFPE_Unknown": "#dbdb8d",
    "GPL96_FF_Unknown": "#f7b6d2",
    "GPL96_FFPE_Unknown": "#e377c2",
    "GPL6244_FF_Unknown": "#e9967a",
    "GPL6244_FFPE_Unknown": "#fa8072",
    "GPL13938_FFPE_Unknown": "#800000",
    "GPL1708_FF_Unknown": "#7f7f7f",
    "GPL14951_FFPE_Unknown": "#c49c94",
    "GPL17077_FF_Unknown": "#c5b0d5",
    "GPL13158_FF_Unknown": "#ad494a",
    "GPL17586_FF_Unknown": "#008b8b",
    "GPL17047_FF_Unknown": "#4682b4",
    "GPL887_FFPE_Unknown": "#b0c4de",
    "GPL10739_FF_Unknown": "#9edae5",
    "GPL8432_FFPE_Unknown": "#006400",
    "GPL20188_FF_Unknown": "#556b2f",
    "GPL23541_FF_Unknown": "#8fbc8f",
    "GPL16686_FF_Unknown": "#333333",
    "GPL26356_FF_Unknown": "#a0522d",
}

strat_pal = {
    "S0_no_removal": (0.6313725490196078, 0.788235294117647, 0.9568627450980393),
    "A_confirmed_bad": (1.0, 0.7058, 0.5098),
    "B_extended_bad": (0.5529, 0.8980, 0.6313),
    "C_rnaseq_only": (1.0, 0.6235, 0.6078),
    "D_malignant_only": (0.8156, 0.7333, 1.0),
    "E1_iterative_r1": (0.8705, 0.7333, 0.6078),
    "E2_iterative_r2": (0.9803, 0.6901, 0.8941),
    "E3_iterative_r3": (0.8117, 0.8117, 0.8117),
    "F_microarray_only": (1.0, 0.9960, 0.6392),
    "G_affymetrix_only": (0.7254, 0.9490, 0.9411),
    "H_affymetrix_extended": (0.741, 0.718, 0.420),
    "I_rare_batches_removed": "#b095c5",
    "J_ff_only": "#7ecbc4",
    "K_ffpe_only": "#f2c27f",
}

IMPUTATION_NAMES = {"strict": "no imputation", "knn": "KNN", "softimpute": "softImpute"}
MISSING_COLOR = "#AAAAAA"

### §0d Plot settings and helpers

In [ ]:
# Plot settings (project CLAUDE.md).
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = GLOBAL_FONT_SIZE
sns.set_style("ticks")

EMBEDDINGS = {"pca": ("PC1", "PC2"), "tsne": ("tSNE1", "tSNE2")}
ROW_TITLES = {"batch": "Batch", "diagnosis": "Diagnosis", "platform": "Platform"}


def level_color(level: str, palette: dict, prefix_match: bool) -> str:
    # exact match for batches / platforms, prefix match for diagnoses; else an MD5 colour
    if pd.isna(level):
        return MISSING_COLOR
    if level in palette:
        return palette[level]
    if prefix_match:
        for key, colour in palette.items():
            if str(level).startswith(key):
                return colour
    colour = "#" + hashlib.md5(str(level).encode()).hexdigest()[:6]
    palette[level] = colour
    print(f"palette: {level} → {colour} (MD5)")
    return colour


def approach_label(strat: str, imp: str, method: str) -> str:
    # plain-word label: "code · strategy, imputation, method"
    method_name = "unharmonized" if method == "01_raw" else rl.method_display_name(method)
    return f"{rl.STRATEGY_DISPLAY_NAMES[strat]}, {IMPUTATION_NAMES[imp]}, {method_name}"

In [ ]:
# Figma units (Daniil 2026-11-09, `daniil_review_261109.md`). Figma imports an SVG with
# 1 SVG pt = 4/3 px, and the figure frame (Figma node 1607-116664) is 756 × 1159 px for the
# 180 × 275 mm page. So every panel is drawn at (slot in Figma px) × 0.75 pt, and every text
# is 7.5 pt = 10 px in Figma. Legends and colour bars ("palettes") may stick out of the slot:
# only the plot area (axes with their tick labels and titles) is fitted to it.
import matplotlib.text as mtext
from matplotlib.transforms import Bbox

FIGMA_FRAME_PX = (756, 1159)
FIGMA_PAGE_MM = (180, 275)
FIGMA_PX_PER_PT = 4 / 3
FIGMA_PX_PER_MM = FIGMA_FRAME_PX[0] / FIGMA_PAGE_MM[0]  # 4.2 px per mm of the page
FIGMA_FONT_PX = 10
FIGMA_FONT_PT = FIGMA_FONT_PX / FIGMA_PX_PER_PT  # 7.5 pt
FIGMA_FIT_LOG = rl.OUTPUT_DIR / "wp14" / "figma_fit" / "nb08_figures_4_5.tsv"
FIGMA_FIT_LOG.parent.mkdir(parents=True, exist_ok=True)

GLOBAL_FONT_SIZE = FIGMA_FONT_PT
SMALL_FONT_SIZE = FIGMA_FONT_PT
plt.rcParams.update({key: FIGMA_FONT_PT for key in (
    "font.size", "axes.labelsize", "axes.titlesize", "xtick.labelsize", "ytick.labelsize",
    "legend.fontsize", "legend.title_fontsize", "figure.titlesize")})


def mm_figsize(width_mm: float, height_mm: float) -> tuple[float, float]:
    """Figure size in inches that lands in Figma at the slot size (page mm → Figma px)."""
    to_inches = FIGMA_PX_PER_MM / FIGMA_PX_PER_PT / 72
    return width_mm * to_inches, height_mm * to_inches


def slot_figsize(stem: str) -> tuple[float, float]:
    """Figure size in inches of the page slot of a panel file (PANEL_SLOTS_MM)."""
    return mm_figsize(*PANEL_SLOTS_MM[stem])


def batch_display_name(batch: str) -> str:
    """Plain batch name: underscores → spaces; microarrays drop the last "Unknown" (library
    preparation is never known for a microarray), RNA-seq keeps its library type."""
    parts = str(batch).split("_")
    if parts[0] != "RNASeq" and len(parts) > 2 and parts[-1] == "Unknown":
        parts = parts[:-1]
    return " ".join(parts)


def is_palette_axis(ax: plt.Axes) -> bool:
    """Colour bars and legend-only axes are palettes: Daniil moves them by hand in Figma."""
    if hasattr(ax, "_colorbar") or getattr(ax, "_figma_palette", False):
        return True
    has_data = bool(ax.lines or ax.collections or ax.images or ax.patches or ax.texts)
    return not ax.axison and not has_data


def plot_extent_px(fig: plt.Figure) -> tuple[float, float]:
    """Width and height in Figma px of the plot area (axes, tick labels, titles, texts on
    the axes, suptitle), without legends and colour bars."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = [ax.get_tightbbox(renderer, bbox_extra_artists=list(ax.texts))
             for ax in fig.axes if ax.get_visible() and not is_palette_axis(ax)]
    boxes += [t.get_window_extent(renderer) for t in fig.texts if t.get_text()]
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text():
        boxes.append(fig._suptitle.get_window_extent(renderer))
    boxes = [b for b in boxes if b is not None and b.width > 0]
    # a legend-only figure has no plot area: measure the whole figure
    union = Bbox.union(boxes) if boxes else fig.bbox
    px_per_display = 72 / fig.dpi * FIGMA_PX_PER_PT
    return union.width * px_per_display, union.height * px_per_display


def set_all_fonts(fig: plt.Figure, size: float) -> None:
    """Every text of the figure at one size (10 px in Figma)."""
    for ax in fig.axes:
        ax.tick_params(labelsize=size)
    for text in fig.findobj(mtext.Text):
        if text.get_text():
            text.set_fontsize(size)


def fit_to_slot(fig: plt.Figure, width_mm: float, height_mm: float | None,
                n_iterations: int = 8) -> None:
    """Resize the figure until its plot area is width_mm wide (and height_mm high; None keeps
    the current aspect ratio of the plot area). Fonts keep their size, so a few redraws are
    needed; fixed-aspect plots (squares) only shrink to fit."""
    target_w = width_mm * FIGMA_PX_PER_MM
    for _ in range(n_iterations):
        plot_w, plot_h = plot_extent_px(fig)
        target_h = plot_h * target_w / plot_w if height_mm is None else height_mm * FIGMA_PX_PER_MM
        if abs(plot_w - target_w) < 1 and abs(plot_h - target_h) < 1:
            break
        fig_w, fig_h = fig.get_size_inches()
        fig.set_size_inches(max(fig_w * target_w / plot_w, 0.5), max(fig_h * target_h / plot_h, 0.5))
    plot_w, plot_h = plot_extent_px(fig)
    target_h = FIGMA_FRAME_PX[1] if height_mm is None else height_mm * FIGMA_PX_PER_MM
    shrink = min(1.0, target_w / plot_w, target_h / plot_h)
    if shrink < 0.995:  # fixed-aspect plots grew in the other direction: scale the whole panel
        fig.set_size_inches(*(fig.get_size_inches() * shrink))


def log_fit(stem: str, fig: plt.Figure, slot_mm: tuple | None) -> None:
    """One row per saved panel: slot and plot area in Figma px (read by r97_figma_check.py)."""
    plot_w, plot_h = plot_extent_px(fig)
    row = pd.DataFrame([{
        "stem": stem,
        "slot_w_px": None if slot_mm is None else round(slot_mm[0] * FIGMA_PX_PER_MM, 1),
        "slot_h_px": None if slot_mm is None or slot_mm[1] is None else round(slot_mm[1] * FIGMA_PX_PER_MM, 1),
        "plot_w_px": round(plot_w, 1), "plot_h_px": round(plot_h, 1),
    }])
    old = pd.read_csv(FIGMA_FIT_LOG, sep="\t") if FIGMA_FIT_LOG.exists() else row.iloc[:0]
    pd.concat([old[old["stem"] != stem], row]).to_csv(FIGMA_FIT_LOG, sep="\t", index=False)


def save_figure(fig: plt.Figure, stem: str, slot_mm: tuple | None = None,
                keep_fonts: bool = False) -> None:
    """Fonts → 10 Figma px (unless keep_fonts: large clustermaps), plot area → slot (default:
    PANEL_SLOTS_MM[stem] when the notebook has it), then SVG + PDF + PNG (200 dpi)."""
    if slot_mm is None and stem in globals().get("PANEL_SLOTS_MM", {}):
        slot_mm = PANEL_SLOTS_MM[stem]
    if not keep_fonts:
        set_all_fonts(fig, FIGMA_FONT_PT)
    if slot_mm is not None:
        fit_to_slot(fig, *slot_mm)
    log_fit(stem, fig, slot_mm)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


# nb08 sizes its panels with MM (inches per page mm): page mm → Figma px.
MM = mm_figsize(1, 1)[0]


## §1 Embeddings and scores

The seven harmonized matrices and their unharmonized counterparts (same strategy and imputation)
are read from the nb07 cache. Marker correlation (L) and cross-batch prediction (N, with the
unharmonized value of the same strategy and imputation) are read from the four-score table, the
same values as in Table 3.

In [ ]:
# Read the cached coordinates (stop with the nb07 command when one is missing).
def cached_coordinates(strat: str, imp: str, method: str) -> pd.DataFrame:
    path = CACHE_DIR / f"{strat}__{imp}__{method}__post0.pkl"
    if not path.exists():
        raise SystemExit(f"{path.name} missing: run nb07 with this approach in its selection")
    return pd.read_pickle(path)


matrices = sorted({m for m in PANELS.values()} | {(s, i, "01_raw") for s, i, _ in PANELS.values()})
coordinates = {m: cached_coordinates(*m) for m in matrices}
sample_platform = rl.load_s0_annotation()["PLATFORM_RNA"].replace({"RNASeq": "RNA-seq"})
for frame in coordinates.values():
    frame["platform"] = sample_platform.reindex(frame.index).values
platform_levels = sorted(sample_platform.dropna().unique(), key=lambda v: (v != "RNA-seq", v))
# r98 audit 2026-11-08: the article platform palette instead of tab20.
platform_palette = {  # article palette, copied from nb06 §0c (RNASeq spelled RNA-seq here)
    "RNA-seq": "#4B0082",
    "GPL96+97": "#FF4500",
    "GPL570": "#FF8C00",
    "GPL96": "#B22222",
    "GPL6244": "#E9967A",
    "GPL13938": "#CD5C5C",
    "GPL1708": "#008080",
    "GPL14951": "#4682B4",
    "GPL17077": "#00CED1",
    "GPL13158": "#1E90FF",
    "GPL17586": "#20B2AA",
    "GPL17047": "#5F9EA0",
    "GPL887": "#B0C4DE",
    "GPL10739": "#2E8B57",
    "GPL20188": "#6B8E23",
    "GPL23541": "#8FBC8F",
    "GPL8432": "#006400",
    "GPL16686": "#8B4513",
    "GPL26356": "#A0522D",
}
assert set(platform_levels) <= set(platform_palette), set(platform_levels) - set(platform_palette)

scores = pd.read_parquet(SCORES_PATH)
scores = scores[~scores["post_rm"].astype(bool)].set_index(["strat", "imp", "method"])
selection = pd.read_csv(SELECTION_PATH, sep="\t")
selection = selection[~selection["post_rm"].astype(str).str.lower().isin(["true", "1"])]
selection = selection.set_index(["strat", "imp", "method"])

panel_rows = []
for panel, (strat, imp, method) in PANELS.items():
    harmonized, raw = coordinates[(strat, imp, method)], coordinates[(strat, imp, "01_raw")]
    is_raw = method == "01_raw"
    if not is_raw:
        assert (strat, imp, method) in selection.index, f"{panel} is not a recommended approach"
    row = scores.loc[(strat, imp, method)]
    raw_row = scores.loc[(strat, imp, "01_raw")]
    panel_rows.append({
        "panel": panel, "strategy": rl.STRATEGY_DISPLAY_NAMES[strat],
        "imputation": IMPUTATION_NAMES[imp],
        "method": "unharmonized" if is_raw else rl.method_display_name(method),
        "n_samples": len(harmonized), "n_genes": harmonized.attrs.get("n_genes"),
        "n_genes_unharmonized": raw.attrs.get("n_genes"),
        # prediction as gated by the rule (Table 3): 3-class, DLBCL vs FL for strategy D
        "marker_correlation": row["L"],
        "prediction_f1": row["N"] if is_raw else selection.loc[(strat, imp, method), "N_gate"],
        "prediction_f1_unharmonized": raw_row["N"] if is_raw else selection.loc[(strat, imp, method), "N_raw"],
        "prediction_classes": 3 if is_raw else int(selection.loc[(strat, imp, method), "gate_N_classes"]),
        "batch_removal": row["batch_removal"], "batch_removal_unharmonized": raw_row["batch_removal"],
        "diagnosis_retention": row["biology_retention"],
        "diagnosis_retention_unharmonized": raw_row["biology_retention"],
        "local_batch_mixing": row["local_batch_scaled"],
        "local_batch_mixing_unharmonized": raw_row["local_batch_scaled"],
        "n_batches": harmonized["batch"].nunique(),
        "pc1_variance_pct": round(100 * harmonized.attrs["pc_variance"][0], 1),
        "pc1_variance_pct_unharmonized": round(100 * raw.attrs["pc_variance"][0], 1),
        "pc1_pc2_variance_pct": round(100 * sum(harmonized.attrs["pc_variance"][:2]), 1),
        "pc1_pc2_variance_pct_unharmonized": round(100 * sum(raw.attrs["pc_variance"][:2]), 1),
    })
panel_table = pd.DataFrame(panel_rows)
panel_table.to_csv(TABLES_DIR / "nb08_figure_4_5_panels.tsv", sep="\t", index=False)
panel_table.round(3)

## §2 Drawing

One file per panel (Daniil assembles the pages in Figma, `figures_svg/PLACEMENT.md`). Panel 4A has
two columns (PCA, tSNE of the unharmonized S0 data); the others four (PCA unharmonized |
harmonized, tSNE unharmonized | harmonized). Points are drawn in one fixed shuffled order so no
batch is always on top.

In [ ]:
# One scatter.
def draw_scatter(ax, frame: pd.DataFrame, colour_by: str, kind: str) -> None:
    x_col, y_col = EMBEDDINGS[kind]
    palette, prefix = {"batch": (rna_batch_palette, False),
                       "diagnosis": (lymphoma_ontogeny_palette, True),
                       "platform": (platform_palette, False)}[colour_by]
    order = np.random.default_rng(SEED).permutation(len(frame))
    shuffled = frame.iloc[order]
    ax.scatter(shuffled[x_col], shuffled[y_col], s=POINT_SIZE, alpha=POINT_ALPHA, linewidths=0,
               c=[level_color(v, palette, prefix) for v in shuffled[colour_by]], rasterized=True)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_box_aspect(1)  # every embedding scatter is a square (Daniil 2026-11-09)
    sns.despine(ax=ax)


def embedding_title(kind: str, frame: pd.DataFrame) -> str:
    # axis names live in the column title (no per-axis labels at print size)
    if kind == "pca":
        variance = frame.attrs["pc_variance"]
        return f"PCA\nPC1 {100 * variance[0]:.0f} %, PC2 {100 * variance[1]:.0f} %"
    return "tSNE"


# Plain diagnosis names, copied from nb01 §0c (r98 consistency review 2026-11-08: Figure 1E and
# Figures 4–5 must name the diagnoses the same way).
DIAGNOSIS_PLAIN_NAMES = {
    "Bone_marrow_CD19+": "Bone marrow CD19+ B cells",
    "Immature": "Immature B cells",
    "Naive": "Naive B cells",
    "GC": "Germinal centre B cells",
    "Centroblast": "Centroblasts",
    "Centrocyte": "Centrocytes",
    "B_cells": "B cells (unsorted)",
    "MZ": "Marginal zone B cells",
    "Memory": "Memory B cells",
    "Plasmablast": "Plasmablasts",
    "Plasma": "Plasma cells",
    "Follicular_Lymphoma": "Follicular lymphoma",
    "Diffuse_Large_B_Cell_Lymphoma": "Diffuse large B-cell lymphoma",
    "Burkitt_Lymphoma": "Burkitt lymphoma",
    "High_Grade_B_Cell_Lymphoma": "High-grade B-cell lymphoma",
}


def legend_handles(frames: list[pd.DataFrame], colour_by: str) -> list:
    levels = sorted(set().union(*[set(f[colour_by].dropna()) for f in frames]),
                    key=lambda v: (colour_by == "platform" and v != "RNA-seq", v))
    palette, prefix = {"batch": (rna_batch_palette, False),
                       "diagnosis": (lymphoma_ontogeny_palette, True),
                       "platform": (platform_palette, False)}[colour_by]
    return [mpatches.Patch(color=level_color(v, palette, prefix),
                           label=DIAGNOSIS_PLAIN_NAMES.get(v, str(v).replace("_", " "))
                           if colour_by == "diagnosis" else batch_display_name(v) if colour_by == "batch"
                           else str(v).replace("_", " "))
            for v in levels]


def score_line(panel: str) -> str:
    row = panel_table.set_index("panel").loc[panel]
    classes = ", DLBCL vs FL" if row.prediction_classes == 2 else ""
    return (f"marker correlation {row.marker_correlation:.2f}\ncross-batch prediction "
            f"(macro-F1{classes}) {row.prediction_f1:.2f} (unharmonized {row.prediction_f1_unharmonized:.2f})")


def panel_rows(panel: str) -> list[str]:
    return ["batch", "diagnosis"] + (["platform"] if panel[0] in PLATFORM_ROW_FIGURES else [])


def pack_columns(fig: plt.Figure, top_axes: list, gap_px: float = 8.0) -> None:
    """Square scatters leave free space between the columns: shrink the panel width until the
    columns (with their titles) are gap_px apart (Figma px). Fonts keep their size."""
    for _ in range(5):
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        boxes = [ax.get_tightbbox(renderer) for ax in top_axes]
        spare = sum(b.x0 - a.x1 for a, b in zip(boxes, boxes[1:])) - gap_px * (len(boxes) - 1) / FIGMA_PX_PER_PT * fig.dpi / 72
        if abs(spare) < 1:  # negative spare = titles collide: widen again
            break
        width, height = fig.get_size_inches()
        fig.set_size_inches(width - 0.7 * spare / fig.dpi, height)


def draw_overview_panel(panel: str) -> plt.Figure:
    # 4A: one row of four scatters (PCA / tSNE × batch / diagnosis) of the unharmonized data
    strat, imp, method = PANELS[panel]
    raw = coordinates[(strat, imp, method)]
    width, height = PANEL_SLOTS_MM[panel]
    fig, axes = plt.subplots(1, 4, figsize=(width * MM, height * MM), layout="constrained")
    for ax, (kind, colour_by) in zip(axes, [("pca", "batch"), ("pca", "diagnosis"),
                                            ("tsne", "batch"), ("tsne", "diagnosis")]):
        draw_scatter(ax, raw, colour_by, kind)
        ax.set_title(f"{embedding_title(kind, raw)}\ncoloured by {ROW_TITLES[colour_by].lower()}",
                     fontsize=SMALL_FONT_SIZE)
    pack_columns(fig, list(axes))
    fig.suptitle(f"{panel[1]}  {approach_label(strat, imp, method)}  ({len(raw):,} samples)",
                 x=0.0, ha="left", fontsize=GLOBAL_FONT_SIZE, fontweight="bold")
    return fig


def draw_pair_panel(panel: str) -> plt.Figure:
    # 4B–5C: rows = colouring, columns = PCA unharmonized | harmonized, tSNE unharmonized | harmonized
    strat, imp, method = PANELS[panel]
    rows = panel_rows(panel)
    raw, harmonized = coordinates[(strat, imp, "01_raw")], coordinates[(strat, imp, method)]
    columns = [("pca", raw), ("pca", harmonized), ("tsne", raw), ("tsne", harmonized)]
    width, height = PANEL_SLOTS_MM[panel]
    fig, axes = plt.subplots(len(rows), len(columns), figsize=(width * MM, height * MM),
                             squeeze=False, layout="constrained")
    for col, (kind, frame) in enumerate(columns):
        state = "harmonized: " + rl.method_display_name(method) if frame is harmonized else "unharmonized"
        axes[0, col].set_title(f"{embedding_title(kind, frame)}\n{state}", fontsize=SMALL_FONT_SIZE)
        for row_index, colour_by in enumerate(rows):
            draw_scatter(axes[row_index, col], frame, colour_by, kind)
    pack_columns(fig, list(axes[0]))
    for row_index, colour_by in enumerate(rows):
        axes[row_index, 0].set_ylabel(ROW_TITLES[colour_by], fontsize=GLOBAL_FONT_SIZE, fontweight="bold")
    fig.suptitle(f"{panel[1]}  {approach_label(strat, imp, method)}  ({len(raw):,} samples)\n"
                 f"{score_line(panel)}", x=0.0, ha="left", fontsize=GLOBAL_FONT_SIZE,
                 fontweight="bold", linespacing=1.3)
    return fig


def draw_legend_figure(figure_number: str) -> plt.Figure:
    # one shared legend per page: union of the levels drawn in that figure's panels
    frames = [coordinates[key] for panel, (s, i, m) in PANELS.items() if panel[0] == figure_number
              for key in {(s, i, m), (s, i, "01_raw")}]
    rows = panel_rows(figure_number + "A")
    width, height = LEGEND_SLOTS_MM[figure_number]
    fig = plt.figure(figsize=(width * MM, height * MM))
    top = 1.0
    for colour_by in rows:
        handles = legend_handles(frames, colour_by)
        ncol = {"batch": 4, "diagnosis": 4, "platform": 7}[colour_by]
        legend = fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, top), ncol=ncol,
                            frameon=False, fontsize=SMALL_FONT_SIZE, title=ROW_TITLES[colour_by],
                            title_fontsize=GLOBAL_FONT_SIZE, handlelength=0.9, handleheight=0.9,
                            columnspacing=0.8, labelspacing=0.25, borderaxespad=0.0)
        legend._legend_box.align = "left"
        fig.canvas.draw()
        box = legend.get_window_extent().transformed(fig.transFigure.inverted())
        top = box.y0 - 0.03
    # legends are palettes (Daniil moves them in Figma): a taller legend only gets a note
    if top < -0.02:
        print(f"note: legend of Figure {figure_number} is taller than its {height} mm slot")
    return fig


def save_panel(fig: plt.Figure, stem: str) -> None:
    # true size: no bbox cropping, so the file is exactly its page slot; 10 px text
    set_all_fonts(fig, FIGMA_FONT_PT)
    slot = {**{PANEL_FILE_STEMS[p]: s for p, s in PANEL_SLOTS_MM.items()},
            **{LEGEND_FILE_STEMS[f]: s for f, s in LEGEND_SLOTS_MM.items()}}[stem]
    log_fit(stem, fig, slot)
    for suffix in ("svg", "pdf", "png"):
        tight = "tight"  # packed panels and legends: the file is exactly its content
        fig.savefig(FIGURES_DIR / f"{stem}.{suffix}", dpi=600 if suffix == "png" else 300,
                    bbox_inches=tight)


for panel in PANELS:
    fig = draw_overview_panel(panel) if PANELS[panel][2] == "01_raw" else draw_pair_panel(panel)
    save_panel(fig, PANEL_FILE_STEMS[panel])
    plt.show() if panel in {"4B", "5A"} else None
    plt.close(fig)
for figure_number, stem in LEGEND_FILE_STEMS.items():
    fig = draw_legend_figure(figure_number)
    save_panel(fig, stem)
    plt.show()
    plt.close(fig)
print("saved", ", ".join([*PANEL_FILE_STEMS.values(), *LEGEND_FILE_STEMS.values()]))


## §3 What the panels show (numbers for the Results text)

The table of §1 (`output/wp9/nb08_figure_4_5_panels.tsv`) holds every number the Results text
quotes for Figures 4–5; r45 reads it. Panel titles carry marker correlation and prediction.

Caveats: (1) the embeddings use only the genes without any missing value in the matrix (column
`n_genes`), as in nb07 and in the submitted figures; (2) tSNE distances between clusters are not
interpretable, only neighbourhoods; (3) the dendrogram panels of the submitted figures are not
redrawn (Daniil, B1 option b).